# 📖 BERT Experiment - Piecewise Linear Attention

Compare **Standard**, **Linear**, and **Piecewise** attention in BERT on SST-2 sentiment classification.

## What This Does

- Trains BERT on SST-2 (Stanford Sentiment Treebank)
- Compares all three attention mechanisms
- Measures training time and classification accuracy
- Generates comparison plots and tables

## Expected Runtime

- **With GPU (T4)**: ~5-10 minutes (3 epochs, tiny model)
- **Without GPU**: ~30-60 minutes (not recommended)

---

## 📦 Setup

In [ ]:
# Configuration
REPO_URL = "https://github.com/grapentt/piecewise-linear-attention.git"
BRANCH = "main"  # Change to your branch if needed

# Clone repository
import os
if not os.path.exists("piecewise-linear-attention"):
    print(f"Cloning repository (branch: {BRANCH})...")
    !git clone -b {BRANCH} {REPO_URL}
else:
    print("Repository already exists")

%cd piecewise-linear-attention
!git pull origin {BRANCH}
print(f"✓ Working directory: {os.getcwd()}")

In [ ]:
# Install dependencies
print("Installing dependencies...")
!pip install -e ".[experiments]" -q
print("✓ Installation complete!")

In [ ]:
# Check GPU
import torch

if torch.cuda.is_available():
    device = "cuda"
    print(f"✓ GPU: {torch.cuda.get_device_name(0)}")
    print(f"  Memory: {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB")
else:
    device = "cpu"
    print("⚠ No GPU - using CPU (will be much slower)")
    print("  Enable GPU: Runtime → Change runtime type → GPU")

print(f"\nDevice: {device}")

---

## 🎯 Experiment Configuration

In [ ]:
# Experiment settings
ATTENTION_TYPES = ["standard", "linear", "piecewise"]  # Which attention types to compare
TASK = "glue/sst2"  # Task (glue/sst2 for sentiment)
MODEL_SIZE = "tiny"  # Model size: tiny or base
EPOCHS = 3  # Number of training epochs
BATCH_SIZE = 32  # Batch size
LEARNING_RATE = 2e-5  # Learning rate

print("Experiment Configuration:")
print(f"  Attention types: {', '.join(ATTENTION_TYPES)}")
print(f"  Task: {TASK}")
print(f"  Model: BERT-{MODEL_SIZE.upper()}")
print(f"  Epochs: {EPOCHS}")
print(f"  Batch size: {BATCH_SIZE}")
print(f"  Learning rate: {LEARNING_RATE}")
print(f"  Device: {device}")

---

## 🚀 Run Experiment

In [ ]:
# Run BERT experiment
!python experiments/bert/train.py \
  --attention-types {' '.join(ATTENTION_TYPES)} \
  --task {TASK} \
  --model-size {MODEL_SIZE} \
  --epochs {EPOCHS} \
  --batch-size {BATCH_SIZE} \
  --lr {LEARNING_RATE} \
  --device {device} \
  --save results/bert/colab_results.json

print("\n" + "="*80)
print("✓ BERT experiment complete!")
print("="*80)

---

## 📊 Visualize Results

In [ ]:
import json
import matplotlib.pyplot as plt
import numpy as np

# Load results
with open("results/bert/colab_results.json") as f:
    results = json.load(f)

# Create figure with 3 subplots
fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# Plot 1: Accuracy over epochs
ax = axes[0]
for result in results:
    attn = result["attention_type"]
    history = result["val_history"]
    accuracies = [epoch["accuracy"] for epoch in history]
    ax.plot(accuracies, label=attn.capitalize(), linewidth=2, marker='o', markersize=6)

ax.set_xlabel("Epoch", fontsize=12)
ax.set_ylabel("Validation Accuracy (%)", fontsize=12)
ax.set_title("Sentiment Classification Accuracy", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3)

# Plot 2: Training time
ax = axes[1]
attention_types = [r["attention_type"].capitalize() for r in results]
train_times = [r["total_train_time"] for r in results]
colors = ["#e74c3c", "#3498db", "#2ecc71"]
bars = ax.bar(attention_types, train_times, color=colors[:len(results)])

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.1f}s', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.set_ylabel("Training Time (seconds)", fontsize=12)
ax.set_title("Training Time", fontsize=14, fontweight="bold")
ax.grid(True, alpha=0.3, axis='y')

# Plot 3: Speedup
ax = axes[2]
baseline_time = next(r["total_train_time"] for r in results if r["attention_type"] == "standard")
speedups = [baseline_time / r["total_train_time"] for r in results]
bars = ax.bar(attention_types, speedups, color=colors[:len(results)])

for bar in bars:
    height = bar.get_height()
    ax.text(bar.get_x() + bar.get_width()/2., height,
            f'{height:.2f}×', ha='center', va='bottom', fontsize=10, fontweight='bold')

ax.axhline(y=1.0, color='red', linestyle='--', alpha=0.5, label='Baseline')
ax.set_ylabel("Speedup vs Standard", fontsize=12)
ax.set_title("Speedup", fontsize=14, fontweight="bold")
ax.legend()
ax.grid(True, alpha=0.3, axis='y')

fig.suptitle(f"BERT Experiment (SST-2 Sentiment Analysis)", fontsize=16, fontweight='bold', y=1.02)
plt.tight_layout()
plt.show()

In [ ]:
# Print summary table
print("\n" + "="*100)
print("RESULTS SUMMARY")
print("="*100)
print(f"{'Attention':<15} {'Accuracy (%)':<15} {'Loss':<12} {'Train Time (s)':<18} {'Speedup':<10}")
print("-"*100)

baseline_time = next(r["total_train_time"] for r in results if r["attention_type"] == "standard")

for result in results:
    attn = result["attention_type"].capitalize()
    acc = result["final_val_accuracy"]
    loss = result["final_val_loss"]
    train_time = result["total_train_time"]
    speedup = baseline_time / train_time
    print(f"{attn:<15} {acc:<15.2f} {loss:<12.4f} {train_time:<18.1f} {speedup:<10.2f}×")

print("="*100)

# Key insights
piecewise = next(r for r in results if r["attention_type"] == "piecewise")
standard = next(r for r in results if r["attention_type"] == "standard")
speedup = baseline_time / piecewise["total_train_time"]
acc_diff = piecewise["final_val_accuracy"] - standard["final_val_accuracy"]

print(f"\n✨ Key Findings:")
print(f"  • Piecewise is {speedup:.2f}× faster than Standard")
print(f"  • Accuracy: {acc_diff:+.2f}% vs Standard")
print(f"  • Trade-off: ~{speedup:.1f}× speedup with ~{abs(acc_diff):.1f}% accuracy change")

---

## 💾 Download Results

In [ ]:
from google.colab import files

print("Downloading results...")
files.download("results/bert/colab_results.json")
print("✓ Download complete!")

---

## 📚 Learn More

- [BERT Experiment Guide](https://github.com/grapentt/piecewise-linear-attention/blob/main/experiments/bert/README.md)
- [Main README](https://github.com/grapentt/piecewise-linear-attention/blob/main/README.md)
- [Theory](https://github.com/grapentt/piecewise-linear-attention/blob/main/THEORY.md)

**Happy Experimenting! 🚀**